# 직접 수집 4 · 결과 — PageRank · 통계 · 시각화

수집·통합한 데이터를 **분석 결과**로 만듭니다.
1. **PageRank** — 네트워크에서 각 문서의 연결 중심성(중요도)
2. **통계 종합** — 편집·조회·링크·정보 → **기술집약도 / 수요부상성 / 공급부상성**
3. **네트워크 시각화**

In [ ]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # pipeline.py 있는 프로젝트 루트 탐색
    if os.path.isfile("pipeline.py"):
        break
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 1                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

In [ ]:
# [이전 단계 재현] 시드 실제 제목 (캐시되어 즉시)
df_true = P.resolve_true_title(SEED, os.path.join(BASE, "true_title.xlsx"))
SEED_TRUE = P.title_space(str(df_true.loc[0, "True_title"]))
print("시드 실제 제목:", SEED_TRUE)

In [ ]:
# [이전 단계 재현] 네트워크 확장 (캐시되어 즉시)
EXPAND = P.expand_network(SEED_TRUE, EXPAND, N_DEPTH)
print("확장 결과:", EXPAND)

In [ ]:
# [이전 단계 재현] 네트워크 필터링 (캐시되어 즉시)
FILTER = P.filter_network(EXPAND, SEED_TRUE, FILTER, mode="balanced")
print("필터 결과:", FILTER)

In [ ]:
# [이전 단계 재현] XTools 수집·통합 (캐시되어 빠름)
P.xtools_collect(SEED_TRUE, FILTER, XTOOLS, FLAG)
outs = P.xtools_integrate(SEED_TRUE, XTOOLS)
print("통합 완료:", list(outs.keys()))

## 1) PageRank — 연결 중심성

In [ ]:
pr_df = P.compute_pagerank(FILTER, PAGERANK, seed_nodes=[SEED_TRUE])
pr_df.sort_values("pagerank", ascending=False).head(10)

## 2) 통계 종합 — 3대 지표
편집/조회/링크/정보 원천을 **기술집약도·수요부상성·공급부상성**으로 환산합니다.

In [ ]:
stats = P.compute_statistics(
    edit_xlsx=outs.get("edit"), pageviews_xlsx=outs.get("pageviews"),
    link_xlsx=outs.get("link"), info_xlsx=outs.get("info"),
    pr_df=pr_df, out_xlsx=STATS, ref_year=None,
)
cols = [c for c in ["title", "기술집약도", "수요부상성", "공급부상성", "확산성", "총편집수", "총조회수"] if c in stats.columns]
stats[cols].sort_values("기술집약도", ascending=False).head(15)

## 3) 네트워크 시각화

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

net = P.format_item_3_network(FILTER, pr_df)   # {nodes:[{id,size,group}], edges:[{source,target}]}
G = nx.Graph()
for n in net["nodes"]:
    G.add_node(n["id"], size=n.get("size", 15))
for e in net["edges"]:
    G.add_edge(e["source"], e["target"])

if G.number_of_nodes():
    pos = nx.spring_layout(G, seed=42, k=1.4 / (G.number_of_nodes() ** 0.5))
    sizes = [G.nodes[n].get("size", 15) * 12 for n in G.nodes()]
    is_seed = [n == SEED_TRUE for n in G.nodes()]
    colors = ["#ff7043" if s else "#42a5f5" for s in is_seed]
    plt.figure(figsize=(12, 9))
    nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.6)
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors, edgecolors="white", linewidths=0.5)
    big = {n: n for n in G.nodes() if G.nodes[n].get("size", 0) > 20 or n == SEED_TRUE}
    nx.draw_networkx_labels(G, pos, labels=big, font_size=8, font_family="Malgun Gothic")
    plt.title(f"{SEED_TRUE} 위키 연관 네트워크 (노드 크기=PageRank)"); plt.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("네트워크가 비어 있습니다.")

**정리**
- 문서별 3대 지표 + PageRank로 **유망도 근거**를 완성했습니다.
- 다음 노트북(**05 보고서**)에서 LLM이 이 결과를 해석합니다.